<a href="https://colab.research.google.com/github/Wilson-G9/Lab5_Bai3_Giac_Ngu./blob/main/Bai3_Giac_Ngu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài 3: Phân loại chất lượng giấc ngủ sinh viên
**Dataset:** student_sleep_patterns.csv | **Thuật toán:** K-Means (Sklearn & Custom)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display, clear_output

## Bước 1: Tải dữ liệu

In [ ]:
from google.colab import files
uploaded = files.upload()   # Upload student_sleep_patterns.csv

df = pd.read_csv('student_sleep_patterns.csv')
print("Shape:", df.shape)
print(df.head())
print(df.info())

## Bước 2: Tiền xử lý

In [ ]:
df_work = df.copy()

drop_cols = ['Student_ID', 'Weekday_Sleep_Start', 'Weekend_Sleep_Start',
             'Weekday_Sleep_End', 'Weekend_Sleep_End']
df_work.drop(columns=[c for c in drop_cols if c in df_work.columns], inplace=True)

if 'Gender' in df_work.columns:
    df_work['Gender'] = LabelEncoder().fit_transform(df_work['Gender'].astype(str))

if 'University_Year' in df_work.columns:
    year_map = {'1st Year':1, '2nd Year':2, '3rd Year':3, '4th Year':4}
    df_work['University_Year'] = df_work['University_Year'].map(year_map)
    df_work['University_Year'].fillna(df_work['University_Year'].median(), inplace=True)

df_work.fillna(df_work.median(numeric_only=True), inplace=True)
print("Sau xử lý NaN:\n", df_work.isnull().sum())
print(df_work.describe())

## Bước 3: Trực quan hóa

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Sleep_Quality'], bins=15, kde=True, color='mediumpurple')
plt.title('Phân phối Sleep Quality'); plt.show()

plt.figure(figsize=(8, 5))
sc = plt.scatter(df_work['Sleep_Duration'], df_work['Sleep_Quality'],
                 c=df_work['Caffeine_Intake'], cmap='YlOrRd', alpha=0.7, s=40)
plt.colorbar(sc, label='Caffeine Intake')
plt.xlabel('Sleep Duration (hours)'); plt.ylabel('Sleep Quality')
plt.title('Sleep Duration vs Sleep Quality'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Study_Hours', 'Physical_Activity', 'Screen_Time']):
    ax.scatter(df_work[col], df_work['Sleep_Quality'], alpha=0.4, s=20)
    ax.set_xlabel(col); ax.set_ylabel('Sleep Quality')
    ax.set_title(f'{col} vs Sleep Quality')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df_work.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Ma trận tương quan'); plt.tight_layout(); plt.show()

## Bước 4: Chuẩn hóa & tìm K tối ưu

In [ ]:
features = ['Sleep_Duration', 'Study_Hours', 'Screen_Time',
            'Caffeine_Intake', 'Physical_Activity']

X = df_work[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias, sil_scores = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(list(K_range), inertias, 'bo-')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method'); ax1.grid(True)
ax2.plot(list(K_range), sil_scores, 'rs-')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score theo K'); ax2.grid(True)
plt.tight_layout(); plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"\n✅ K tối ưu: {best_k}  (Silhouette = {max(sil_scores):.4f})")

## Bước 5a: K-Means dùng Sklearn

In [ ]:
km_sklearn = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_sklearn.fit(X_scaled)
labels_sk = km_sklearn.labels_
sil_sk    = silhouette_score(X_scaled, labels_sk)

print(f"[Sklearn K-Means]  Silhouette: {sil_sk:.4f} | Inertia: {km_sklearn.inertia_:.2f}")

df_cluster = pd.DataFrame(X_scaled, columns=features)
df_cluster['Cluster_Sklearn'] = labels_sk
df_cluster['Sleep_Quality']   = df_work['Sleep_Quality'].values

print("\nTrung bình theo cụm (Sklearn):")
print(df_cluster.groupby('Cluster_Sklearn')[features + ['Sleep_Quality']].mean().round(2))

## Bước 5b: K-Means tự cài đặt (không dùng Sklearn)

In [ ]:
class KMeansCustom:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters   = n_clusters
        self.max_iter     = max_iter
        self.tol          = tol
        self.random_state = random_state

    def fit(self, X):
        np.random.seed(self.random_state)
        idx = np.random.choice(len(X), self.n_clusters, replace=False)
        self.centroids_ = X[idx].copy()
        for i in range(self.max_iter):
            dists = np.array([np.linalg.norm(X - c, axis=1)
                              for c in self.centroids_])
            self.labels_ = np.argmin(dists, axis=0)
            new_c = np.array([
                X[self.labels_ == k].mean(axis=0)
                if np.any(self.labels_ == k) else self.centroids_[k]
                for k in range(self.n_clusters)
            ])
            if np.linalg.norm(new_c - self.centroids_) < self.tol:
                print(f"  Hội tụ sau {i+1} vòng lặp")
                break
            self.centroids_ = new_c
        self.inertia_ = float(sum(
            np.sum((X[self.labels_ == k] - self.centroids_[k])**2)
            for k in range(self.n_clusters)
        ))
        return self

    def predict(self, X):
        dists = np.array([np.linalg.norm(X - c, axis=1)
                          for c in self.centroids_])
        return np.argmin(dists, axis=0)

km_custom = KMeansCustom(n_clusters=best_k, random_state=42)
km_custom.fit(X_scaled)
labels_cu = km_custom.labels_
sil_cu    = silhouette_score(X_scaled, labels_cu)

print(f"[Custom K-Means]   Silhouette: {sil_cu:.4f} | Inertia: {km_custom.inertia_:.2f}")

df_cluster['Cluster_Custom'] = labels_cu
print("\nTrung bình theo cụm (Custom):")
print(df_cluster.groupby('Cluster_Custom')[features + ['Sleep_Quality']].mean().round(2))

## Bước 6: So sánh Sklearn vs Custom

In [ ]:
ari = adjusted_rand_score(labels_sk, labels_cu)
print("======= SO SÁNH K-MEANS =======")
print(f"{'':30} {'Sklearn':>10} {'Custom':>10}")
print("-" * 52)
print(f"{'Silhouette Score':30} {sil_sk:>10.4f} {sil_cu:>10.4f}")
print(f"{'Inertia':30} {km_sklearn.inertia_:>10.2f} {km_custom.inertia_:>10.2f}")
print(f"\nAdjusted Rand Index: {ari:.4f}  (1.0 = hoàn toàn giống nhau)")

plt.figure(figsize=(6, 4))
bars = plt.bar(['Sklearn', 'Custom'], [sil_sk, sil_cu],
               color=['steelblue', 'coral'], edgecolor='black')
for bar, v in zip(bars, [sil_sk, sil_cu]):
    plt.text(bar.get_x()+bar.get_width()/2, v+0.002,
             f'{v:.4f}', ha='center', fontweight='bold')
plt.ylabel('Silhouette Score')
plt.title('So sánh Silhouette Score: Sklearn vs Custom')
plt.tight_layout(); plt.show()

# PCA 2D
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for k in range(best_k):
    mask = labels_sk == k
    ax1.scatter(X_pca[mask,0], X_pca[mask,1], label=f'Cụm {k}', alpha=0.6, s=30)
ax1.set_title(f'Sklearn K-Means (k={best_k})'); ax1.legend(); ax1.grid(True, alpha=0.3)
for k in range(best_k):
    mask = labels_cu == k
    ax2.scatter(X_pca[mask,0], X_pca[mask,1], label=f'Cụm {k}', alpha=0.6, s=30)
ax2.set_title(f'Custom K-Means (k={best_k})'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Bước 7: Giao diện nhập liệu dự đoán cụm

In [ ]:
sleep_w  = widgets.FloatText(value=7.0, step=0.5,  description='Sleep Hours:')
study_w  = widgets.FloatText(value=6.0, step=0.5,  description='Study Hours:')
screen_w = widgets.FloatText(value=3.0, step=0.5,  description='Screen Time:')
caff_w   = widgets.IntSlider(value=2,   min=0, max=10, description='Caffeine:')
phys_w   = widgets.IntSlider(value=30,  min=0, max=120, description='Physical(min):')
btn      = widgets.Button(description='🔍 Dự đoán Cụm', button_style='primary')
out      = widgets.Output()

def on_predict(b):
    with out:
        clear_output()
        vals = np.array([[sleep_w.value, study_w.value, screen_w.value,
                          caff_w.value,  phys_w.value]])
        scaled = scaler.transform(vals)
        c_sk = km_sklearn.predict(scaled)[0]
        c_cu = km_custom.predict(scaled)[0]
        info = df_cluster.groupby('Cluster_Sklearn')[features + ['Sleep_Quality']].mean()
        sq_mean = info.loc[c_sk, 'Sleep_Quality']
        print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(f"✅ Sklearn  → Cụm {c_sk}")
        print(f"✅ Custom   → Cụm {c_cu}")
        print(f"\n📊 Sleep Quality TB của Cụm {c_sk}: {sq_mean:.2f}/10")
        if sq_mean >= 7:
            print("💤 Giấc ngủ TỐT")
        elif sq_mean >= 5:
            print("😐 Giấc ngủ TRUNG BÌNH")
        else:
            print("😴 Giấc ngủ KÉM – cần cải thiện!")

btn.on_click(on_predict)
display(widgets.VBox([
    widgets.HTML("<h3>🔍 Nhập thông tin sinh viên</h3>"),
    sleep_w, study_w, screen_w, caff_w, phys_w,
    btn, out
]))